In [2]:
from time import perf_counter

import jax
jax.config.update("jax_enable_x64", True)

from jax import numpy as jnp

import equinox as eqx
import optax

from agp.models import ResidualRBM
from agp.sampler import SpinSampler
from agp.operator import LocalOperator, TransverseFieldIsing
from agp.graph import Cube
from agp.jacobian import natural_gradient
from agp.utils import abs2, param_count, model_params

import matplotlib.pyplot as plt

## NQS setup

Ground state initialization

In [3]:
shape = (8, 8)

$$
H = - J \sum _{\langle i j \rangle} \sigma^z _i \sigma^z _j - h \sum _i \sigma^x _i
$$

In [4]:
graph = Cube(shape, pbc=True)

In [5]:
key = jax.random.PRNGKey(1337)

In [6]:
H = TransverseFieldIsing(graph, J=1.0, h=0.8)

In [7]:
# logpsi = ConvRBM(graph.shape, pbc=True, kernel_size=5, n_channels=2, key=key)
# logpsi = ConvRBM(graph.shape, n_channels=4, key=key)
logpsi = ResidualRBM(graph.shape, n_channels=4, n_blocks=2, key=key)

In [8]:
param_count(logpsi)

4132

In [9]:
logpsi

ResidualRBM(
  encoder=ResidualEncoder(
    embedding=SpinEmbedding(
      conv=Conv(
        num_spatial_dims=2,
        weight=f64[4,1,1,1],
        bias=None,
        in_channels=1,
        out_channels=4,
        kernel_size=(1, 1),
        stride=(1, 1),
        padding=((0, 0), (0, 0)),
        dilation=(1, 1),
        groups=1,
        use_bias=False,
        padding_mode='ZEROS'
      )
    ),
    blocks=[
      ResidualBlock(
        conv1=Conv(
          num_spatial_dims=2,
          weight=f64[8,4,3,3],
          bias=f64[8,1,1],
          in_channels=4,
          out_channels=8,
          kernel_size=(3, 3),
          stride=(1, 1),
          padding='SAME',
          dilation=(1, 1),
          groups=1,
          use_bias=True,
          padding_mode='CIRCULAR'
        ),
        conv2=Conv(
          num_spatial_dims=2,
          weight=f64[4,8,3,3],
          bias=f64[4,1,1],
          in_channels=8,
          out_channels=4,
          kernel_size=(3, 3),
          strid

In [10]:
sampler = SpinSampler(dims=shape, n_samples=64, n_chains=4)

In [11]:
key, = jax.random.split(key, 1)
samples = sampler(logpsi, key).reshape(-1, *graph.shape)
samples.shape

(256, 8, 8)

In [12]:
logpsi(samples[0])

Array(-0.00030099+0.00045806j, dtype=complex128)

In [13]:
def cosine_schedule(max_val, min_val, n_steps, start=0.0, end=1.0):

    assert 0.0 <= start < end <= 1.0, "Start and end must be in the range [0, 1] and start < end."
    start_step, end_step = int(start * n_steps), int(end * n_steps)

    def schedule(step):
        scaled_step = (step - start_step) / (end_step - start_step)
        scaled_step = jnp.clip(scaled_step, 0.0, 1.0)
        return min_val + (max_val - min_val) * jnp.cos(jnp.pi * scaled_step / 2) ** 2

    return schedule

In [14]:
n_steps = 500
schedule = cosine_schedule(1e-2, 2e-4, n_steps, start=0.0, end=0.9)
optim = optax.sgd(schedule)
opt_state = optim.init(model_params(logpsi))

In [15]:
@eqx.filter_jit
def step(logpsi, opt_state, key):

    samples = sampler(logpsi, key).reshape(-1, *graph.shape)

    local_energy = LocalOperator(H, logpsi)
    eloc = jax.vmap(local_energy)(samples)

    energy_val = jnp.mean(eloc)
    energy_var = jnp.mean(abs2(eloc - energy_val))
    energy_val = jnp.real(energy_val)
    
    grads = natural_gradient(logpsi, samples, eloc, diag_shift=1e-5)

    updates, opt_state = optim.update(grads, opt_state)
    logpsi = eqx.apply_updates(logpsi, updates)

    return energy_val, energy_var, logpsi, opt_state

In [ ]:
energies, variances = [], []
clock = perf_counter()

for i in range(n_steps):

    key = jax.random.fold_in(key, i)

    energy, var, logpsi, opt_state = step(logpsi, opt_state, key)

    energies.append(energy.item())
    variances.append(var.item())

    if perf_counter() - clock > 4:
        lr = schedule(i)
        print(f"Step {i+1:4} | Energy: {energy:.4e} | Variance: {var:.4e} | LR: {lr:.2e}")
        clock = perf_counter()

Step    1 | Energy: -5.1950e+01 | Variance: 1.3770e+02 | LR: 1.00e-02
Step    2 | Energy: -4.5055e+01 | Variance: 2.0812e+02 | LR: 1.00e-02
Step    3 | Energy: -4.7015e+01 | Variance: 1.8054e+02 | LR: 1.00e-02


In [ ]:
plt.plot(energies, label="VMC energy")
# plt.axhline(E0, color="black", linestyle="--", label="Exact energy")
plt.legend()

plt.xlabel("Iteration")
plt.ylabel("Energy")

In [ ]:
plt.semilogy(variances)
plt.xlabel("Iteration")
plt.ylabel("Variance")